<a href="https://colab.research.google.com/github/DhairyaDave08/BetaData-SpaceX/blob/main/Notebooks/03_Baseline__Modelling.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Baseline Model (Logistic Regression)**

Goal: establish a simple, interpretable baseline before moving to XGBoost.
Uses an era-stratified split (train on earlier launches, test on the most
recent ones) rather than a random split, to respect the "no future leakage"
constraint and mirror how this model would actually be used.

In [14]:
!git clone https://github.com/DhairyaDave08/BetaData-SpaceX.git
%cd BetaData-SpaceX
!pip install -q -r requirements.txt

import sys
sys.path.append('/content/BetaData-SpaceX')

import pandas as pd
import numpy as np
from sklearn.metrics import (
    roc_auc_score, average_precision_score, brier_score_loss,
    f1_score, classification_report
)

Cloning into 'BetaData-SpaceX'...
remote: Enumerating objects: 94, done.
remote: Counting objects: 100% (94/94), done.
remote: Compressing objects: 100% (78/78), done.
remote: Total 94 (delta 37), reused 25 (delta 6), pack-reused 0 (from 0)
Receiving objects: 100% (94/94), 509.28 KiB | 11.07 MiB/s, done.
Resolving deltas: 100% (37/37), done.
/content/BetaData-SpaceX/BetaData-SpaceX/BetaData-SpaceX


In [15]:
import pandas as pd

# This is the Raw URL from GitHub
url = "https://raw.githubusercontent.com/DhairyaDave08/BetaData-SpaceX/main/Data/processed/features.csv"

# Load the data
features = pd.read_csv(url, parse_dates=["launch_date"])

print(f"Successfully loaded dataset with shape: {features.shape}")
features.head(3)

Successfully loaded dataset with shape: (4198, 20)


,launch_date,year,decade,mission_success,rocket_family_grouped,launch_site_grouped,country_grouped,payload_capacity_kg,payload_capacity_known,vehicle_prior_flights,vehicle_prior_success_rate,site_prior_flights,site_prior_success_rate,country_prior_flights,country_prior_success_rate,vehicle_age_days,weather_available,wind_speed_max_kmh,temp_max_c,precipitation_mm
0,1957-10-04 19:28:00,1957.0,1950.0,1,other,Site 1/5,Kazakhstan,10750,False,0,0.903764,0,0.903764,0,0.903764,0,True,13.0,29.1,0.0
1,1957-11-03 02:30:00,1957.0,1950.0,1,other,Site 1/5,Kazakhstan,10750,False,1,1.000000,1,1.000000,1,1.000000,29,True,15.9,12.2,0.0
2,1957-12-06 16:44:00,1957.0,1950.0,0,Vanguard,LC-18A,USA,10750,False,0,0.903764,0,0.903764,0,0.903764,0,False,NaN,NaN,NaN


## Era-stratified train/test split

We split by time: the earliest ~80% of launches become the training set,
the most recent ~20% become the test set. This is deliberately harder than
a random split — the model must generalize to more recent vehicles and
practices, not just interpolate within a shuffled mix of eras.

In [17]:
from src.train import era_stratified_split, get_X_y

train_df, test_df = era_stratified_split(features, test_size=0.2)

Train: 1957-10-04 to 2007-07-02 (3358 rows, 89.4% success rate)
Test:  2007-07-05 to 2020-08-07 (840 rows, 94.3% success rate)


In [18]:
from src.train import build_logistic_pipeline

X_train, y_train = get_X_y(train_df)
X_test, y_test = get_X_y(test_df)

baseline_pipeline = build_logistic_pipeline()
baseline_pipeline.fit(X_train, y_train)

print("Baseline Logistic Regression trained.")

Baseline Logistic Regression trained.


/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


In [19]:
probs = baseline_pipeline.predict_proba(X_test)[:, 1]
preds = (probs >= 0.5).astype(int)

print("=== BASELINE METRICS (held-out test set) ===")
print(f"ROC-AUC:      {roc_auc_score(y_test, probs):.4f}")
print(f"PR-AUC:       {average_precision_score(y_test, probs):.4f}")
print(f"Brier score:  {brier_score_loss(y_test, probs):.4f}")
print(f"F1:           {f1_score(y_test, preds):.4f}")
print()
print(classification_report(y_test, preds, target_names=["Failure", "Success"]))

=== BASELINE METRICS (held-out test set) ===
ROC-AUC:      0.7257
PR-AUC:       0.9707
Brier score:  0.1575
F1:           0.8583

              precision    recall  f1-score   support

     Failure       0.13      0.54      0.20        48
     Success       0.97      0.77      0.86       792

    accuracy                           0.76       840
   macro avg       0.55      0.66      0.53       840
weighted avg       0.92      0.76      0.82       840



## Interpretation

With a 90.4% success rate baseline, a model that always predicts "success"
would get 90.4% accuracy while being useless — exactly the trap the problem
statement warns against. We report ROC-AUC, PR-AUC, Brier score, and F1
instead of relying on accuracy, and used `class_weight="balanced"` in
Logistic Regression to avoid the model trivially collapsing to majority-class
prediction.

This baseline gives us a reference point. Next: `04_modeling_final.ipynb`
builds an XGBoost model, applies isotonic calibration, and compares against
this baseline.